# Phase 6 · S3 · arch 1 (decoder) · Houlsby adapters · part-wise summarisation
**Houlsby adapters** (bottleneck 128 after attention and feed-forward in every block, plus the layer norms;
backbone frozen) on the arch-1 decoder. Its tokenizer needs ~2.3 tokens per word, so a part has 622 prompt tokens and up
to 400 summary tokens. arch 1 has no KV cache, so generation recomputes the sequence (slower).
Submit `submission_p6_s3_arch1_houlsby.zip`.

**Method (all Phase-6 notebooks):** long judgments are handled part-wise. Document and reference summary are split into
the same number *n* of ordered parts (n = summary length / part size, at most 8; at test time predicted from the document
length with a log-log fit on training data). Each document part is compressed to the prompt budget by keeping evenly
spaced sentences (start, middle and end stay represented) and trained to generate the matching summary part. Generation:
greedy, no repeated 3-grams, repetition penalty 1.15, minimum length, `</s>` stops; parts are concatenated and repeated
sentences removed. Validation = 5% of the training documents (no dev split is provided); ROUGE is computed locally,
the leaderboard computes BERTScore. Only the Phase-1 models and the Phase-6 training data are used.

**Run:** GPU **T4 x2** (cell 1 must print `GPUs: 2`), Internet **On**, Secret `capstone`, Add Input -> **`phase2-backbones`**.
**Save Version -> Save & Run All.** Never submit a `smoke_*` zip.

In [ ]:
# 1 · Settings
import os, sys, json, glob, time, random, math, gc
import numpy as np, torch
BASE = dict(
    batch=8, accum=2, epochs=3, weight_decay=0.01, seed=1337, budget_hours=6.0,   # training (val loss each epoch)
    n_max=8,                                   # at most 8 document/summary parts per document
    min_frac=0.5, no_repeat=3, rep_pen=1.15, gen_batch=16, n_val_gen=60,           # decoding + validation generation
)
TAG = "S3"
PLAN = [dict(BASE, run="p6_s3_arch1_houlsby", arch="arch1_decoder", method="houlsby", bottleneck=128, lr=1e-3, G=400)]
OUT = "/kaggle/working"
SMOKE_CHECK = True
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
found = glob.glob("/kaggle/input/**/phase2-backbones/manifest.json", recursive=True) or \
        glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert found, "attach the dataset phase2-backbones (Add Input -> Your Datasets)"
ROOT = os.path.dirname(found[0])
print("backbones:", ROOT, "| GPUs:", torch.cuda.device_count(), "| CPUs:", os.cpu_count(), "| runs:", [c["run"] for c in PLAN])
if torch.cuda.device_count() == 0:
    raise SystemExit("NO GPU in this session: set Accelerator = GPU T4 x2 (or this account's GPU quota is used up)")

In [ ]:
# 2 · Hugging Face token (Kaggle secret "capstone" or "HF_TOKEN")
from huggingface_hub import login, whoami
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for _n in ("capstone", "HF_TOKEN"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(_n)
            if HF_TOKEN: break
        except Exception:
            pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "add the Hugging Face token as a Kaggle secret and attach it"
login(token=HF_TOKEN); print("Hugging Face user:", whoami(token=HF_TOKEN)["name"])

## 3 · Code
`ner_model.py` (LoRA / LoRMA / adapter modules from Phase 2), `ner_data.py` (its helper), `sm.py` (Phase 6: parts, models, training, distillation, generation, ROUGE, submission).

In [ ]:
os.makedirs('p6src', exist_ok=True)

In [ ]:
%%writefile p6src/ner_data.py
# Phase 2 (L-NER) data utilities (pure Python, no torch):
#   gold parsing, words with exact character offsets, word-level BIO tags and span targets,
#   span decoding strategies, document-level label propagation, strict scoring, error analysis, submission.
import bisect
import re
from collections import Counter, defaultdict

# ======================================================================================================
# 1. Gold annotations
# ======================================================================================================
_START = ("start", "start_char", "start_offset", "begin", "s")
_END = ("end", "end_char", "end_offset", "stop", "e")
_LABEL = ("label", "type", "entity_type", "tag", "category")
_TEXT = ("text", "entity_text", "span", "mention", "value")
_IDS = ("id", "doc_id", "document_id", "docid")
_ENTS = ("entities", "annotations", "spans", "labels", "gold")


def _pick(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None


def _entity(item):
    """one annotation in any common shape -> (start, end, label, text or None)"""
    if isinstance(item, (list, tuple)):
        s, e, lab = item[0], item[1], item[2]
        txt = item[3] if len(item) > 3 else None
    elif isinstance(item, dict):
        s, e, lab, txt = _pick(item, _START), _pick(item, _END), _pick(item, _LABEL), _pick(item, _TEXT)
    else:
        raise ValueError(f"unknown annotation format: {item!r}")
    return int(s), int(e), str(lab), (txt if isinstance(txt, str) else None)


def parse_gold(raw):
    """gold json (dict doc -> list, dict doc -> {entities: [...]}, or list of records) -> {doc: [(s,e,lab,txt)]}"""
    out = {}
    if isinstance(raw, list):
        for rec in raw:
            out[str(_pick(rec, _IDS))] = [_entity(x) for x in (_pick(rec, _ENTS) or [])]
        return out
    for doc, v in raw.items():
        if isinstance(v, dict):
            v = _pick(v, _ENTS) or []
        out[str(doc)] = [_entity(x) for x in v]
    return out


def parse_texts(raw):
    """text json -> {doc: text}"""
    if isinstance(raw, list):
        return {str(_pick(r, _IDS)): _pick(r, ("text", "content", "document")) for r in raw}
    return {str(k): (v if isinstance(v, str) else _pick(v, ("text", "content", "document"))) for k, v in raw.items()}


def end_convention(texts, gold):
    """Is `end` exclusive (text[s:e] == entity text) or inclusive (text[s:e+1])? Decided by the data."""
    exc = inc = total = 0
    bad = []
    for doc, ents in gold.items():
        t = texts.get(doc)
        if t is None:
            continue
        for s, e, lab, txt in ents:
            if txt is None:
                continue
            total += 1
            if t[s:e] == txt:
                exc += 1
            elif t[s:e + 1] == txt:
                inc += 1
            elif len(bad) < 5:
                bad.append((doc, s, e, lab, txt, t[s:e + 1]))
    conv = "inclusive" if inc > exc else "exclusive"
    return conv, dict(total_with_text=total, exclusive_match=exc, inclusive_match=inc, examples_no_match=bad)


def to_exclusive(gold, conv):
    """-> {doc: [(s, e_exclusive, label)]} (internal convention: python slice)"""
    add = 1 if conv == "inclusive" else 0
    return {d: [(s, e + add, lab) for s, e, lab, _ in ents] for d, ents in gold.items()}


# ======================================================================================================
# 2. Tokens and words with character offsets
# ======================================================================================================
# The pre-tokenisation decides where an entity can start or end. "bpe" = the split used in pretraining
# (a word keeps its punctuation: "Singh," is one piece). "fine" = punctuation is split off words, so
# "Singh" | "," can be cut exactly. Every piece is still encoded by the same BPE (same vocabulary).
SPLITS = {
    ("space", "bpe"): re.compile(r" ?\S+|\s+(?!\S)|\s+"),
    ("space", "fine"): re.compile(r" ?[^\W_]+| ?_+| ?[^\w\s]|\s+(?!\S)|\s+"),
    ("a1", "bpe"): re.compile(r"\S+|\s+"),
    ("a1", "fine"): re.compile(r"[^\W_]+|_+|[^\w\s]|\s+"),
}


def tokenizer_kind(tok):
    return "space" if getattr(tok, "space_prefix", False) else "a1"


def encode_words(tok, text, split="fine"):
    """text -> ids, wtok, wcs, wce
    ids  : BPE token ids of the whole text (decode(ids) == text)
    words: every pre-token piece that is not pure whitespace. For word w:
           wtok[w] = index of its first non-whitespace token (the token that represents the word),
           [wcs[w], wce[w]) = its characters without surrounding whitespace."""
    pat = SPLITS[(tokenizer_kind(tok), split)]
    ids, wtok, wcs, wce = [], [], [], []
    covered = 0
    for m in pat.finditer(text):
        chunk, base = m.group(), m.start()
        assert base == covered, f"pre-tokeniser skipped characters at {covered}"
        covered = m.end()
        char_of_byte = []
        for ci, ch in enumerate(chunk):
            char_of_byte.extend([ci] * len(ch.encode("utf-8")))
        pos, first, ws, we = 0, None, None, None
        for t in tok.encode_chunk(chunk.encode("utf-8")):
            n = len(tok.itos[t])
            s, e = base + char_of_byte[pos], base + char_of_byte[pos + n - 1] + 1
            pos += n
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                if first is None:
                    first, ws = len(ids), s
                we = e
            ids.append(t)
        if first is not None:
            wtok.append(first); wcs.append(ws); wce.append(we)
    assert covered == len(text), "pre-tokeniser did not cover the whole text"
    return ids, wtok, wcs, wce


# ======================================================================================================
# 3. Targets: BIO tags per word, and (start word, end word, label) for the span head
# ======================================================================================================
IGNORE = -100


def tag_vocab(labels):
    tags = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    return tags, {t: i for i, t in enumerate(tags)}


def word_targets(wcs, wce, spans, tag2id, label2id):
    """gold char spans -> (BIO tag per word, [(first_word, last_word, label_id)], #overlapping spans skipped)"""
    n = len(wcs)
    tags, taken, wspans, conflicts = [0] * n, [False] * n, [], 0
    for s, e, lab in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = bisect.bisect_right(wce, s)              # first word ending after s
        hi = bisect.bisect_left(wcs, e, lo)           # first word starting at/after e
        if lo >= hi:
            continue
        if any(taken[lo:hi]):
            conflicts += 1
            continue
        for k in range(lo, hi):
            tags[k] = tag2id[("B-" if k == lo else "I-") + lab]
            taken[k] = True
        wspans.append((lo, hi - 1, label2id[lab]))
    return tags, wspans, conflicts


def allowed_transitions(tags):
    """BIO validity: allowed[i][j] = may tag j follow tag i; start_ok[j] = may a sequence start with j."""
    K = len(tags)
    ok = [[True] * K for _ in range(K)]
    start_ok = [not t.startswith("I-") for t in tags]
    for i, a in enumerate(tags):
        for j, b in enumerate(tags):
            if b.startswith("I-") and not (a != "O" and a[2:] == b[2:]):
                ok[i][j] = False                      # I-X only after B-X or I-X
    return ok, start_ok


# ======================================================================================================
# 4. Decoding
# ======================================================================================================
def spans_from_tags(tags, wcs, wce, id2tag):
    """word tags -> [(s, e, label)]. B-X starts a span; I-X continues a span of type X; an I-X after O or
    after another type also starts a span (lenient repair, used with greedy argmax decoding)."""
    out, cur = [], None
    for t, s, e in zip(tags, wcs, wce):
        tag = id2tag[t]
        if tag == "O":
            if cur: out.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [s, e, lab]
        else:
            cur[1] = e
    if cur: out.append(cur)
    return [tuple(x) for x in out]


def spans_from_candidates(cands, wcs, wce, id2label):
    """span head: cands = {(i, j): (prob, label_id)} (label_id 0 = none) -> non-overlapping spans,
    highest probability first (greedy)."""
    used, out = set(), []
    for (i, j), (p, l) in sorted(cands.items(), key=lambda kv: -kv[1][0]):
        if l == 0 or any(k in used for k in range(i, j + 1)):
            continue
        used.update(range(i, j + 1))
        out.append((wcs[i], wce[j], id2label[l]))
    return sorted(out)


_WORDCH = re.compile(r"\w")


def propagate(spans, text, min_chars=4):
    """Document-level consistency: a string predicted as label L somewhere in the document is also
    labelled L at its other (unlabelled, whole-word) occurrences. Majority label per string."""
    votes = defaultdict(Counter)
    for s, e, l in spans:
        t = text[s:e]
        if len(t) >= min_chars and _WORDCH.search(t):
            votes[t][l] += 1
    covered = sorted((s, e) for s, e, _ in spans)
    starts = [s for s, _ in covered]

    def free(s, e):
        k = bisect.bisect_left(starts, e)
        return not any(cs < e and ce > s for cs, ce in covered[max(0, k - 50):k])

    out = list(spans)
    for t, c in votes.items():
        lab = c.most_common(1)[0][0]
        i = text.find(t)
        while i >= 0:
            j = i + len(t)
            boundary = (i == 0 or not text[i - 1].isalnum()) and (j == len(text) or not text[j].isalnum())
            if boundary and free(i, j):
                out.append((i, j, lab))
                k = bisect.bisect_left(starts, i); covered.insert(k, (i, j)); starts.insert(k, i)
            i = text.find(t, i + 1)
    return sorted(set(out))


def windows(n, size, stride):
    """window start positions covering [0, n); the last one is pulled back to end at n, so every window
    (except in documents shorter than size) has exactly `size` tokens - no padding needed."""
    if n <= size:
        return [0]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return starts


# ======================================================================================================
# 5. Strict scoring and error analysis
# ======================================================================================================
def strict_scores(pred, gold, labels):
    """pred/gold: {doc: [(s,e,label)]} -> macro/micro P/R/F1 (exact start, end and label) + per label."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        rows[l] = dict(P=P, R=R, F1=F, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    used = [l for l in labels if rows[l]["support"] or rows[l]["fp"]]
    mac = lambda k: sum(rows[l][k] for l in used) / max(1, len(used))
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro = 2 * T / (2 * T + FP + FN) if T else 0.0
    return dict(macro_F1=mac("F1"), macro_P=mac("P"), macro_R=mac("R"), micro_F1=micro, per_label=rows)


def error_breakdown(pred, gold):
    """every mistake classified: wrong label on an exact span, boundary error (overlap, same label),
    boundary+label error, missed entity (no overlapping prediction), spurious prediction."""
    c = Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        c["correct"] += len(p & g)
        ps, used = sorted(p - g), set()
        for s, e, l in sorted(g - p):
            over = [x for x in ps if x[0] < e and x[1] > s]
            if not over:
                c["missed"] += 1; continue
            used.update(over)
            if any(o[0] == s and o[1] == e for o in over): c["wrong_label_exact_span"] += 1
            elif any(o[2] == l for o in over): c["boundary_error_same_label"] += 1
            else: c["boundary_and_label_error"] += 1
        c["spurious"] += len([x for x in ps if x not in used])
    return dict(c)


def label_confusion(pred, gold):
    """exact-span matches with a different label: Counter[(gold label, predicted label)]"""
    c = Counter()
    for doc in gold:
        pm = {(s, e): l for s, e, l in pred.get(doc, [])}
        for s, e, l in gold[doc]:
            if (s, e) in pm and pm[(s, e)] != l:
                c[(l, pm[(s, e)])] += 1
    return c


# ======================================================================================================
# 6. Submission
# ======================================================================================================
def to_submission(pred, conv, texts=None):
    """{doc: [(s, e_exclusive, label)]} -> {doc: [[s, e, label(, text)]]} in the dataset's own end convention"""
    sub = 1 if conv == "inclusive" else 0
    out = {}
    for doc, spans in pred.items():
        rows = []
        for s, e, l in sorted(spans):
            row = [int(s), int(e - sub), l]
            if texts is not None: row.append(texts[doc][s:e])
            rows.append(row)
        out[doc] = rows
    return out

In [ ]:
%%writefile p6src/ner_model.py
# Phase 2 (L-NER) model side:
#   Phase-1 backbones as feature extractors (all 4 architectures, bidirectional or causal),
#   fine-tuning / parameter-efficient methods (full, partial, BitFit, LoRA, LoRMA, Houlsby/Pfeiffer adapters,
#   linear probe), prediction heads (linear softmax, CRF, span classifier), knowledge distillation,
#   sliding-window training and document-level prediction.
import bisect
import math
import os
import pickle
import random
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_data as nd

ARCH_INFO = {   # folder -> (tokenizer file, longest sequence the backbone supports)
    "arch1_decoder": ("a1_bpe8000.pkl", 1024),
    "arch2_encdec": ("a1_bpe8000.pkl", 768),
    "arch3_prefixlm": ("space_bpe8000.pkl", 1024),
    "arch4_mla": ("space_bpe8000.pkl", 1024),
}
NEG = -1e4


def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available())


def _add_code_path(root):
    code = os.path.join(root, "code")
    if code not in sys.path:
        sys.path.insert(0, code)


def load_tokenizer(arch, root):
    _add_code_path(root)
    import a1_tokenizer, space_bpe  # noqa: F401  (classes needed to unpickle)
    return pickle.load(open(os.path.join(root, "tokenizers", ARCH_INFO[arch][0]), "rb"))


# ======================================================================================================
# 1. Backbones
# ======================================================================================================
def _mla_forward(self, x, cos, sin, cache=None):
    """MLA attention of architecture 4 without the causal mask (used when self.bidirectional is True)."""
    if cache is not None or not getattr(self, "bidirectional", False):
        return self._causal_forward(x, cos, sin, cache)
    from rope import apply_rope
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2)
    q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
    q_rope = apply_rope(q_rope, cos.to(q.dtype), sin.to(q.dtype))
    c_kv = self.kv_norm(self.kv_down(x))
    k_r = apply_rope(self.k_rope(x)[:, None], cos.to(q.dtype), sin.to(q.dtype))
    k_nope = self.k_up(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
    v = self.v_up(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
    q = torch.cat([q_nope, q_rope], dim=-1)
    k = torch.cat([k_nope, k_r.expand(B, self.h, T, self.rope)], dim=-1)
    out = F.scaled_dot_product_attention(q, k, v, is_causal=False, dropout_p=self.dropout if self.training else 0.0)
    return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), None


class Backbone(nn.Module):
    """Phase-1 language model reduced to a feature extractor: token ids (B,T) -> hidden states (B,T,d).
    bidirectional: drop the causal mask (arch 1, 4), use the whole window as prefix (arch 3); the arch-2
    encoder is bidirectional by design. keep_layers: keep only the first k blocks (distillation student).
    init="random": same architecture, random weights (control experiment for transfer from Phase 1)."""

    def __init__(self, arch, root, bidirectional=True, keep_layers=None, init="pretrained"):
        super().__init__()
        _add_code_path(root)
        self.arch, self.bidirectional = arch, bidirectional
        ck = torch.load(os.path.join(root, arch, "ckpt_best.pt"), map_location="cpu", weights_only=False)
        cfg = dict(ck["cfg"])
        if arch == "arch4_mla":
            from mla import MLAConfig, MLALM, MLAttention
            if not hasattr(MLAttention, "_causal_forward"):
                MLAttention._causal_forward = MLAttention.forward
                MLAttention.forward = _mla_forward
            m = MLALM(MLAConfig(**cfg))
        elif arch == "arch3_prefixlm":
            from prefixlm import PrefixLMConfig, PrefixLM
            m = PrefixLM(PrefixLMConfig(**cfg))
        elif arch == "arch2_encdec":
            from encdec import EncDecConfig, EncDecLM
            m = EncDecLM(EncDecConfig(**cfg))
        elif arch == "arch1_decoder":
            from model_arch import LMConfig, DecoderLM
            m = DecoderLM(LMConfig(**cfg))
        else:
            raise ValueError(arch)
        if init == "pretrained":
            missing, unexpected = m.load_state_dict(ck["model"], strict=False)
            assert not unexpected and all(x in ("rope", "pos_emb", "mask") or "rope" in x or "mask" in x
                                          or "pos_emb" in x for x in missing), (missing[:5], unexpected[:5])
        else:
            for mod in m.modules():
                if isinstance(mod, (nn.Linear, nn.Embedding)):
                    nn.init.normal_(mod.weight, std=0.02)
                    if getattr(mod, "bias", None) is not None: nn.init.zeros_(mod.bias)
        self.phase1_step, self.phase1_val = ck.get("step"), ck.get("val_loss")
        if arch == "arch2_encdec":                        # NER uses only the (bidirectional) encoder
            m.decoder = nn.ModuleList(); m.dec_norm = nn.Identity()
        if arch == "arch1_decoder":                       # A3 attention keeps its causal mask in a buffer
            for blk in m.blocks:
                blk.attn.causal_mask.fill_(True) if bidirectional else None
        if arch == "arch4_mla":
            for blk in m.blocks:
                blk.attn.bidirectional = bidirectional
        if keep_layers:
            blocks = self._blocks(m)
            del blocks[keep_layers:]
        m.head = None if arch != "arch2_encdec" else m.head   # LM head is not used (tied to the embedding)
        self.m, self.d, self.max_len = m, cfg["d_model"], ARCH_INFO[arch][1]

    @staticmethod
    def _blocks(m):
        return m.encoder if hasattr(m, "encoder") else m.blocks

    @property
    def blocks(self):
        return self._blocks(self.m)

    @property
    def final_norm(self):
        m = self.m
        return m.enc_norm if hasattr(m, "enc_norm") else (m.ln_f if hasattr(m, "ln_f") else m.norm)

    def forward(self, ids):
        m = self.m
        if self.arch == "arch4_mla":
            return m.hidden(ids)
        if self.arch == "arch3_prefixlm":
            return m.hidden(ids, prefix_len=ids.size(1) if self.bidirectional else 0)
        if self.arch == "arch2_encdec":
            return m.encode(ids)
        B, T = ids.shape                                  # arch 1: DecoderLM.forward without the LM head
        x = m.tok_emb(ids)
        if m.cfg.scale_emb:
            x = x * math.sqrt(m.cfg.d_model)
        if m.cfg.pos_emb_type == "sinusoidal":
            x = x + m.pos_emb[:T].unsqueeze(0)
        for blk in m.blocks:
            x = blk(x)
        return m.ln_f(x)


# ======================================================================================================
# 2. Fine-tuning / adaptation methods
# ======================================================================================================
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): W x + (alpha/r) * B A x, with W frozen, A random, B = 0 (starts at W)."""

    def __init__(self, base, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (low-rank MULTIPLICATIVE adaptation): W' = (I + (alpha/r) B A) W, with W frozen.
    The identity term gives the product full rank ("rank inflation" by identity), and B = 0 makes the
    adapted layer start exactly at the pretrained one. Computed as h + s * B(A h) with h = W x, so the
    d_out x d_in product (BA)W is never formed. A bias, if any, is added after the multiplicative update."""

    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + self.scale * F.linear(F.linear(h, self.A), self.B)
        return h if self.base.bias is None else h + self.base.bias


class Adapter(nn.Module):
    """bottleneck adapter (Houlsby et al. 2019): x + up(GELU(down(x))), up = 0 at start (identity)."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))


class AdapterWrap(nn.Module):
    """applies an adapter to the output of a sub-layer (attention or feed-forward), before the residual add"""

    def __init__(self, module, d, bottleneck):
        super().__init__()
        self.module, self.adapter = module, Adapter(d, bottleneck)

    def __getattr__(self, name):                      # keep attributes of the wrapped module reachable
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["module"], name)

    def forward(self, *args, **kwargs):
        out = self.module(*args, **kwargs)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def _replace_linears(root_module, make):
    n = 0
    for name, child in list(root_module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(root_module, name, make(child)); n += 1
        else:
            n += _replace_linears(child, make)
    return n


def apply_method(net, cfg):
    """Freeze/unfreeze and insert adaptation modules according to cfg["method"]. The task head is always
    trained. Returns a short description for the record."""
    bb, method = net.backbone, cfg["method"]
    for p in bb.parameters():
        p.requires_grad = method == "full_ft"
    desc = method
    if method == "partial":                           # layer-wise: only the top k blocks + final norm
        k = cfg.get("train_top_layers", 4)
        for blk in list(bb.blocks)[-k:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in bb.final_norm.parameters(): p.requires_grad = True
        desc = f"partial (top {k} of {len(bb.blocks)} blocks)"
    elif method == "bitfit":                          # only bias terms (Ben Zaken et al. 2022)
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True; n += p.numel()
        assert n > 0, "BitFit needs bias terms - this backbone has none (use arch 1 or arch 2)"
        desc = f"bitfit ({n:,} bias parameters)"
    elif method in ("lora", "lorma"):
        r, a = cfg.get("rank", 16), cfg.get("alpha", 32)
        make = (lambda lin: LoRALinear(lin, r, a, cfg.get("lora_dropout", 0.05))) if method == "lora" \
            else (lambda lin: LoRMALinear(lin, r, a))
        n = sum(_replace_linears(blk, make) for blk in bb.blocks)
        for mod in bb.modules():
            if isinstance(mod, (LoRALinear, LoRMALinear)):
                mod.A.requires_grad = True; mod.B.requires_grad = True
        desc = f"{method} (rank {r}, alpha {a}, {n} linear layers)"
    elif method in ("houlsby", "pfeiffer"):           # Houlsby: after attention and FFN; Pfeiffer: after FFN only
        bott = cfg.get("bottleneck", 64)
        for blk in bb.blocks:
            if method == "houlsby":
                blk.attn = AdapterWrap(blk.attn, bb.d, bott)
            blk.ffn = AdapterWrap(blk.ffn, bb.d, bott)
        for mod in bb.modules():
            if isinstance(mod, Adapter):
                for p in mod.parameters(): p.requires_grad = True
            if isinstance(mod, (nn.LayerNorm,)) or type(mod).__name__ == "RMSNorm":
                for p in mod.parameters(): p.requires_grad = True     # Houlsby also trains the norms
        desc = f"{method} adapters (bottleneck {bott})"
    elif method == "probe":                           # frozen backbone, only the head (linear probe)
        desc = "linear probe (frozen backbone)"
    elif method != "full_ft":
        raise ValueError(method)
    return desc


def count_params(net):
    total = sum(p.numel() for p in net.parameters())
    train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return total, train


# ======================================================================================================
# 3. Prediction heads
# ======================================================================================================
class CRF(nn.Module):
    """linear-chain CRF over word tags. Learned transitions + fixed BIO constraints (forbidden moves = -1e4)."""

    def __init__(self, tags):
        super().__init__()
        K = len(tags)
        ok, start_ok = nd.allowed_transitions(tags)
        self.trans = nn.Parameter(torch.zeros(K, K)); self.start = nn.Parameter(torch.zeros(K))
        self.end = nn.Parameter(torch.zeros(K))
        self.register_buffer("ban", (~torch.tensor(ok)).float() * NEG)
        self.register_buffer("ban_start", (~torch.tensor(start_ok)).float() * NEG)

    def scores(self, train=False):
        # training windows may start inside an entity (first word I-X), so the start constraint is only
        # applied when decoding whole documents
        return self.trans + self.ban, (self.start if train else self.start + self.ban_start), self.end

    def nll(self, em, tags, mask):
        """em (B,W,K) float, tags (B,W) (pads ignored), mask (B,W) bool, prefix-contiguous"""
        trans, start, end = self.scores(train=True)
        B, W, K = em.shape
        tags = tags.clamp(min=0)
        mf = mask.float()
        # score of the gold path
        gold = start[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, W):
            step = trans[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            gold = gold + step * mf[:, t]
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        gold = gold + end[last]
        # log partition
        alpha = start[None] + em[:, 0]
        for t in range(1, W):
            nxt = torch.logsumexp(alpha[:, :, None] + trans[None], dim=1) + em[:, t]
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        logz = torch.logsumexp(alpha + end[None], dim=1)
        has = mask[:, 0]
        return ((logz - gold) * has.float()).sum() / has.float().sum().clamp(min=1)


def viterbi(em, trans, start, end):
    """em (n,K) -> best tag sequence (list) under transition scores (CPU)"""
    n, K = em.shape
    if n == 0:
        return []
    score = start + em[0]
    back = []
    for t in range(1, n):
        s = score[:, None] + trans
        best, idx = s.max(0)
        score = best + em[t]; back.append(idx)
    score = score + end
    tag = int(score.argmax())
    path = [tag]
    for idx in reversed(back):
        tag = int(idx[tag]); path.append(tag)
    return path[::-1]


class NERModel(nn.Module):
    """backbone -> word representations (first token of each word) -> head.
    head = "linear": softmax over BIO tags (cross-entropy, optional class weights)
           "crf":    BIO tag emissions + CRF (sequence-level likelihood)
           "span":   classify every word span up to max_span_words (start/end projections, bilinear term,
                     width embedding), label 0 = not an entity."""

    def __init__(self, backbone, tags, labels, head="linear", dropout=0.2, class_weights=None,
                 max_span_words=32, span_dim=256, neg_weight=0.5):
        super().__init__()
        self.backbone, self.head_type, self.tags = backbone, head, tags
        self.id2label = {i + 1: l for i, l in enumerate(labels)}          # span head: class 0 = no entity
        n_labels = len(labels)
        d, K = backbone.d, len(tags)
        self.drop = nn.Dropout(dropout)
        if head in ("linear", "crf"):
            self.head = nn.Linear(d, K)
            nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
            self.register_buffer("class_weights", class_weights if class_weights is not None else torch.ones(K))
            if head == "crf":
                self.crf = CRF(tags)
        elif head == "span":
            C = n_labels + 1
            self.maxw = max_span_words
            self.head = nn.ModuleDict(dict(
                s=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()), e=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()),
                ls=nn.Linear(span_dim, C), le=nn.Linear(span_dim, C), bil=nn.Linear(span_dim, C, bias=False),
                width=nn.Embedding(max_span_words, C)))
            w = torch.ones(C); w[0] = neg_weight
            self.register_buffer("class_weights", w)
        else:
            raise ValueError(head)

    def word_states(self, ids, wpos):
        h = self.backbone(ids)                                            # (B,T,d)
        return h.gather(1, wpos[:, :, None].expand(-1, -1, h.size(-1)))   # (B,W,d)

    def span_logits(self, hw, wmask):
        H = self.head
        s, e = H["s"](hw), H["e"](hw)                                     # (B,W,p)
        B, W, p = s.shape
        k = torch.arange(self.maxw, device=hw.device)
        j = torch.arange(W, device=hw.device)[:, None] + k[None]          # (W,maxw)
        valid = (j < W)[None] & wmask[:, :, None] & wmask.gather(1, j.clamp(max=W - 1).reshape(1, -1).expand(B, -1)).view(B, W, -1)
        ej = e.gather(1, j.clamp(max=W - 1).reshape(1, -1, 1).expand(B, -1, p)).view(B, W, self.maxw, p)
        logits = H["ls"](s)[:, :, None] + H["le"](ej) + H["bil"](s[:, :, None] * ej) + H["width"](k)[None, None]
        return logits.float(), valid

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        with autocast():
            hw = self.drop(self.word_states(ids, wpos))
            if self.head_type == "span":
                logits, valid = self.span_logits(hw, wmask)
            else:
                logits = self.head(hw).float()
        if self.head_type == "span":
            if stgt is None:                                   # invalid slots -> certainly "no entity"
                inv = ~valid
                out = logits.masked_fill(inv[..., None], NEG)
                out[..., 0] = out[..., 0].masked_fill(inv, 0.0)
                return out
            tgt = stgt.masked_fill(~valid, nd.IGNORE)
            return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                                   weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)
        if wtags is None:
            return logits
        if self.head_type == "crf":
            return self.crf.nll(logits, wtags, wmask).reshape(1)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), wtags.reshape(-1),
                               weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)


class KDWrapper(nn.Module):
    """Knowledge distillation (Hinton et al. 2015) at word level:
    loss = a * CE(student, gold) + (1 - a) * T^2 * KL(teacher_T || student_T). The teacher is frozen."""

    def __init__(self, student, teacher, T=2.0, alpha=0.5):
        super().__init__()
        self.student, self.teacher, self.T, self.alpha = student, teacher, T, alpha
        for p in teacher.parameters(): p.requires_grad = False

    def train(self, mode=True):
        super().train(mode); self.teacher.eval(); return self

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        if wtags is None:
            return self.student(ids, wpos, wmask)
        with torch.no_grad():
            t = self.teacher(ids, wpos, wmask)                            # (B,W,K)
        s = self.student(ids, wpos, wmask)
        m = wmask.reshape(-1)
        ce = F.cross_entropy(s.reshape(-1, s.size(-1)), wtags.reshape(-1), ignore_index=nd.IGNORE)
        sl, tl = s.reshape(-1, s.size(-1))[m] / self.T, t.reshape(-1, t.size(-1))[m] / self.T
        kl = F.kl_div(F.log_softmax(sl, -1), F.softmax(tl, -1), reduction="batchmean") * self.T ** 2
        return (self.alpha * ce + (1 - self.alpha) * kl).reshape(1)


# ======================================================================================================
# 4. Documents -> windows -> batches
# ======================================================================================================
def prepare_docs(texts, gold, tok, tag2id, label2id, split="fine"):
    docs, conflicts = {}, 0
    for d, t in texts.items():
        ids, wtok, wcs, wce = nd.encode_words(tok, t, split)
        r = dict(ids=ids, wtok=wtok, wcs=wcs, wce=wce)
        if gold is not None:
            r["wtags"], r["wspans"], c = nd.word_targets(wcs, wce, gold.get(d, []), tag2id, label2id); conflicts += c
        docs[d] = r
    return docs, conflicts


class Windowing:
    """Cuts every document into windows of `size` tokens. With header_tokens > 0, the first tokens of the
    document (case title, parties, counsel) are put in front of every window, followed by a separator,
    so each window can see who the appellant/respondent/counsel are. Header words are never labelled."""

    def __init__(self, tok, size, header_tokens=0, max_len=1024):
        self.sep = tok.encode("\n\n") if header_tokens else []
        self.H = header_tokens
        self.size = min(size, max_len - (header_tokens + len(self.sep) if header_tokens else 0))
        assert self.size >= 64, "window too small"

    def windows(self, docs, stride):
        out = []
        for d, r in docs.items():
            if r["wtok"]:
                for s in nd.windows(len(r["ids"]), self.size, stride):
                    out.append((d, s, min(self.size, len(r["ids"]))))
        return out

    def window_words(self, r, s, L):
        w0 = bisect.bisect_left(r["wtok"], s); w1 = bisect.bisect_left(r["wtok"], s + L)
        return w0, w1

    def batch(self, docs, b, device, head=None, maxw=None):
        ids, wpos, wtags, ranges, stgts = [], [], [], [], []
        for d, s, L in b:
            r = docs[d]
            head_ids = (r["ids"][:self.H] + self.sep) if self.H else []
            off = len(head_ids) - s
            ids.append(head_ids + r["ids"][s:s + L])
            w0, w1 = self.window_words(r, s, L)
            ranges.append((w0, w1))
            wpos.append([r["wtok"][w] + off for w in range(w0, w1)])
            if "wtags" in r:
                wtags.append(r["wtags"][w0:w1])
        T = len(ids[0]); assert all(len(x) == T for x in ids)
        Wm = max(1, max(len(x) for x in wpos))
        P = torch.zeros(len(b), Wm, dtype=torch.long); M = torch.zeros(len(b), Wm, dtype=torch.bool)
        for i, x in enumerate(wpos):
            P[i, :len(x)] = torch.tensor(x, dtype=torch.long); M[i, :len(x)] = True
        out = dict(ids=torch.tensor(ids, dtype=torch.long), wpos=P, wmask=M)
        if wtags:
            Tg = torch.full((len(b), Wm), nd.IGNORE, dtype=torch.long)
            for i, x in enumerate(wtags):
                Tg[i, :len(x)] = torch.tensor(x, dtype=torch.long)
            out["wtags"] = Tg
            if head == "span":
                S = torch.zeros(len(b), Wm, maxw, dtype=torch.long)
                for i, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
                    for gi, gj, l in docs[d]["wspans"]:
                        if gj < w0 or gi >= w1:
                            continue
                        if gi >= w0 and gj < w1:                    # entity inside the window
                            if gj - gi < maxw: S[i, gi - w0, gj - gi] = l
                        elif gi >= w0:                              # runs past the window end: unknown
                            S[i, gi - w0, :] = nd.IGNORE
                        else:                                       # started before the window: unknown
                            for a in range(max(w0, gj - maxw + 1), gj + 1):
                                S[i, a - w0, gj - a] = nd.IGNORE
                out["stgt"] = S
        return {k: v.to(device) for k, v in out.items()}, ranges


def length_batches(win, bs, shuffle, rng=None):
    by_len = {}
    for w in win:
        by_len.setdefault(w[2], []).append(w)
    out = []
    for _, ws in sorted(by_len.items()):
        if shuffle: rng.shuffle(ws)
        out += [ws[i:i + bs] for i in range(0, len(ws), bs)]
    if shuffle: rng.shuffle(out)
    return out


# ======================================================================================================
# 5. Prediction: windows -> document-level scores -> spans
# ======================================================================================================
@torch.no_grad()
def predict_scores(net, docs, wd, stride, bs, device):
    """returns per document: linear -> averaged tag probabilities (n_words,K); crf -> averaged emissions;
    span -> {(i,j): averaged class probabilities}. Overlapping windows are combined with a tent weight
    (a word in the middle of a window has more context than one at its edge)."""
    net.eval()
    model = net.student if isinstance(net, KDWrapper) else net
    head = model.head_type
    out = {}
    for b in length_batches(wd.windows(docs, stride), bs, shuffle=False):
        batch, ranges = wd.batch(docs, b, device)
        res = net(batch["ids"], batch["wpos"], batch["wmask"]).float().cpu()
        for k, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
            n = w1 - w0
            if n == 0:
                continue
            r = docs[d]
            pos = torch.tensor([r["wtok"][w] - s for w in range(w0, w1)], dtype=torch.float)
            wt = 1 + torch.minimum(pos, L - 1 - pos)
            if head == "span":
                cand = out.setdefault(d, {})
                pr = res[k, :n].softmax(-1)                                 # (n, maxw, C)
                keep = (pr[..., 1:].max(-1).values > 0.2).nonzero().tolist()     # plausible entities only
                for i, kk in keep:
                    j = i + kk
                    if j >= n: continue
                    key = (w0 + i, w0 + j); ww = float(min(wt[i], wt[j]))
                    acc = cand.get(key)
                    cand[key] = [pr[i, kk] * ww, ww] if acc is None else [acc[0] + pr[i, kk] * ww, acc[1] + ww]
            else:
                if d not in out:
                    nw = len(r["wtok"]); out[d] = [torch.zeros(nw, res.size(-1)), torch.zeros(nw, 1)]
                v = res[k, :n].softmax(-1) if head == "linear" else res[k, :n]
                out[d][0][w0:w1] += v * wt[:, None]; out[d][1][w0:w1] += wt[:, None]
    final = {}
    for d in docs:
        if head == "span":
            final[d] = {key: a / w for key, (a, w) in out.get(d, {}).items()}
        else:
            v = out.get(d)
            final[d] = (v[0] / v[1].clamp(min=1e-9)) if v is not None else torch.zeros(0, len(model.tags))
    return final


def decode(scores, docs, model, texts, strategy="viterbi", propagate=False):
    """document scores -> {doc: [(s, e, label)]}.
    linear head: "greedy" (argmax per word + lenient BIO repair) or "viterbi" (best BIO-valid sequence
    under the word log-probabilities); crf head: Viterbi with the learned transitions; span head: greedy
    non-overlapping spans by probability. propagate: document-level label consistency (see ner_data)."""
    head, tags = model.head_type, model.tags
    id2tag = dict(enumerate(tags))
    pred = {}
    if head != "span":
        if head == "crf":
            trans, start, end = [x.detach().float().cpu() for x in model.crf.scores()]
        else:
            ok, start_ok = nd.allowed_transitions(tags)
            trans = (~torch.tensor(ok)).float() * NEG; start = (~torch.tensor(start_ok)).float() * NEG
            end = torch.zeros(len(tags))
    for d, sc in scores.items():
        r = docs[d]
        if head == "span":
            cands = {k: (float(v[1:].max()), int(v.argmax())) for k, v in sc.items()}
            spans = nd.spans_from_candidates(cands, r["wcs"], r["wce"], model.id2label)
        elif head == "linear" and strategy == "greedy":
            spans = nd.spans_from_tags(sc.argmax(-1).tolist(), r["wcs"], r["wce"], id2tag)
        else:
            em = sc.clamp(min=1e-9).log() if head == "linear" else sc
            spans = nd.spans_from_tags(viterbi(em, trans, start, end), r["wcs"], r["wce"], id2tag)
        pred[d] = nd.propagate(spans, texts[d]) if propagate else spans
    return pred


def decoding_options(head):
    base = ["greedy", "viterbi"] if head == "linear" else ["viterbi"] if head == "crf" else ["greedy"]
    return [(s, p) for s in base for p in (False, True)]


# ======================================================================================================
# 6. Training
# ======================================================================================================
def train(net, train_docs, val_docs, val_texts, val_gold, labels, wd, cfg, device, log=print):
    """AdamW (backbone/adaptation LR + head LR), linear warm-up and decay, fp16, gradient accumulation,
    DataParallel over all GPUs; evaluates strict macro-F1 on the validation documents after every epoch
    and keeps the best weights (early stopping with patience)."""
    rng = random.Random(cfg["seed"])
    model = net.student if isinstance(net, KDWrapper) else net
    gpus = torch.cuda.device_count()
    para = nn.DataParallel(net) if gpus > 1 else net
    is_head = lambda n: n.startswith(("head.", "crf.", "student.head.", "student.crf."))
    named = [(n, p) for n, p in net.named_parameters() if p.requires_grad]
    groups = [dict(params=[p for n, p in named if not is_head(n)], lr=cfg["lr"]),
              dict(params=[p for n, p in named if is_head(n)], lr=cfg["lr_head"])]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    win = wd.windows(train_docs, cfg["train_stride"])
    acc = cfg.get("accum", 1)
    steps_per_epoch = math.ceil(len(win) / cfg["batch_size"] / acc)
    total = steps_per_epoch * cfg["epochs"]; warm = max(1, int(cfg["warmup_frac"] * total))
    log(f"train windows {len(win):,} of {wd.size} tokens (+{wd.H} header) | {steps_per_epoch} steps/epoch x "
        f"{cfg['epochs']} epochs | trainable {sum(p.numel() for p in params):,} | GPUs {max(gpus, 1)}")
    best, best_state, history, step, bad, t0 = -1.0, None, [], 0, 0, time.time()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    maxw = getattr(model, "maxw", None)
    for ep in range(1, cfg["epochs"] + 1):
        net.train(); run, n = 0.0, 0
        bl = length_batches(win, cfg["batch_size"], shuffle=True, rng=rng)
        for bi, b in enumerate(bl):
            batch, _ = wd.batch(train_docs, b, device, model.head_type, maxw)
            loss = para(**batch).mean() / acc
            scaler.scale(loss).backward()
            run += loss.item() * acc; n += 1
            if (bi + 1) % acc and bi + 1 < len(bl):
                continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if step % cfg["log_every"] == 0:
                log(f"  ep {ep} step {step}/{total} | loss {run / n:.4f} | {(time.time() - t0) / 60:.1f} min")
                run, n = 0.0, 0
        if cfg.get("select_last"):                    # refit on all documents: fixed epochs, no selection
            history.append(dict(epoch=ep, step=step, minutes=round((time.time() - t0) / 60, 1)))
            log(f"epoch {ep} done (refit on all documents: no validation-based selection)")
            continue
        sc = predict_scores(net, val_docs, wd, cfg["eval_stride"], cfg["eval_batch"], device)
        strategy = "viterbi" if model.head_type != "span" else "greedy"
        res = nd.strict_scores(decode(sc, val_docs, model, val_texts, strategy), val_gold, labels)
        history.append(dict(epoch=ep, step=step, macro_F1=res["macro_F1"], micro_F1=res["micro_F1"],
                            macro_P=res["macro_P"], macro_R=res["macro_R"], minutes=round((time.time() - t0) / 60, 1)))
        note = ""
        if res["macro_F1"] > best + 1e-4:
            best, bad, note = res["macro_F1"], 0, "  <- best"
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in net.state_dict().items()}
        else:
            bad += 1
        log(f"epoch {ep}: val strict macro-F1 {res['macro_F1']:.4f} | micro {res['micro_F1']:.4f} | "
            f"P {res['macro_P']:.3f} R {res['macro_R']:.3f}{note}")
        if cfg.get("patience") and bad >= cfg["patience"]:
            log(f"early stopping: no improvement for {bad} epochs"); break
        if (time.time() - t0) / 3600 > cfg["max_train_hours"]:
            log("time budget reached - stopping"); break
    if best_state is not None:
        net.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    return dict(best_val_macro_F1=best, history=history, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2))


def trainable_state(net):
    """only the trained tensors (what a PEFT method has to store per task)"""
    names = {n for n, p in net.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in net.state_dict().items() if k in names}


@torch.no_grad()
def self_test(net, wd, docs, device, head, maxw=None):
    """one forward + backward on two real windows before the long run (catches shape/device errors early)"""
    b = wd.windows(docs, wd.size)[:2]
    if len(b) == 2 and b[0][2] != b[1][2]: b = b[:1]
    batch, _ = wd.batch(docs, b, device, head, maxw)
    net.train()
    with torch.enable_grad():
        loss = net(**batch).mean(); loss.backward()
    net.zero_grad(set_to_none=True)
    out = net(batch["ids"], batch["wpos"], batch["wmask"])
    return float(loss), tuple(out.shape)

In [ ]:
%%writefile p6src/sm.py
# Phase 6 (abstractive legal summarisation) on the Phase-1 language models.
#   Long documents: document and reference summary are split into the same number n of ordered parts (n from the
#   summary length; at test time from the document length, fitted on training data). Part i of the document,
#   compressed to fit the context (evenly spaced sentences), is trained to generate part i of the summary.
#   Decoder-only models (arch 1, 3, 4): [part header + document part + " SUMMARY:"] -> summary part + </s>, loss on the
#   summary tokens only (arch 3: the prompt is a bidirectional prefix). arch 2: encoder = document part, decoder = summary.
#   Generation: greedy + no-repeat-3-gram + repetition penalty, minimum length, </s> stops; parts are concatenated
#   and duplicate sentences removed.  Adaptation: LoRMA / LoRA / Houlsby adapters / full FT; distillation from a
#   teacher's token distributions (same tokenizer).
import json
import math
import os
import pickle
import random
import re
import sys
import time
import zipfile
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_model as nm

VERSION = "p6-v1"
PAD, UNK, BOS, EOS = 0, 1, 2, 3
INFO = {  # arch -> (tokenizer, context)
    "arch1_decoder": ("a1_bpe8000.pkl", 1024), "arch2_encdec": ("a1_bpe8000.pkl", 768),
    "arch3_prefixlm": ("space_bpe8000.pkl", 1024), "arch4_mla": ("space_bpe8000.pkl", 1024)}

# ======================================================================================================
# 1. Data
# ======================================================================================================
def read_json_any(path):
    with open(path, encoding="utf-8") as f:
        head = f.read(1)
    if head in "[{":
        try:
            return json.load(open(path, encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]


def parse(raw):
    if isinstance(raw, dict):
        for k in ("data", "examples", "docs"):
            if k in raw and isinstance(raw[k], (list, dict)):
                raw = raw[k]; break
    if isinstance(raw, dict):
        raw = [dict(v, id=v.get("id", k)) if isinstance(v, dict) else dict(id=k, document=v) for k, v in raw.items()]
    out = []
    for i, d in enumerate(raw):
        did = next((d[k] for k in ("id", "doc_id", "document_id", "case_id", "name") if k in d), i)
        doc = next((d[k] for k in ("document", "text", "judgment", "doc", "source") if k in d), "")
        summ = next((d[k] for k in ("summary", "abstract", "target", "reference") if k in d), None)
        if isinstance(doc, list): doc = " ".join(map(str, doc))
        if isinstance(summ, list): summ = " ".join(map(str, summ))
        out.append(dict(id=did, document=" ".join(str(doc).split()), summary=None if summ is None else " ".join(str(summ).split())))
    return out


_SENT = re.compile(r"(?<=[.!?;])\s+(?=[A-Z0-9(\"'\[])")


def sentences(text, min_chars=25):
    out = []
    for s in _SENT.split(text):
        s = s.strip()
        if not s: continue
        if out and len(s) < min_chars: out[-1] += " " + s
        else: out.append(s)
    return out or [text[:2000]]


_TOK = None


def _init(tokfile, root):
    global _TOK
    sys.path.insert(0, os.path.join(root, "code"))
    import a1_tokenizer, space_bpe  # noqa: F401
    _TOK = pickle.load(open(os.path.join(root, "tokenizers", tokfile), "rb"))


def _enc(text):
    if getattr(_TOK, "space_prefix", False):
        return _TOK.encode(" " + text)
    return _TOK.encode(" ") + _TOK.encode(text)


def _encode_doc(args):
    doc, summ, max_sents = args
    ds = [np.asarray(_enc(s), dtype=np.int32) for s in sentences(doc)[:max_sents]]
    ss = None if summ is None else [np.asarray(_enc(s), dtype=np.int32) for s in sentences(summ)]
    return ds, ss


class Tok:
    """tokenizer handle in the main process (+ multiprocessing tokenisation of whole datasets)"""

    def __init__(self, arch, root):
        self.arch, self.root, self.file = arch, root, INFO[arch][0]
        _init(self.file, root); self.tok = _TOK

    def enc(self, text):
        return _enc(text)

    def dec(self, ids):
        return self.tok.decode([int(i) for i in ids if int(i) > EOS]).strip()

    def docs(self, docs, workers=4, max_sents=4000, log=print):
        t0 = time.time()
        args = [(d["document"], d["summary"], max_sents) for d in docs]
        if workers > 1:
            import multiprocessing as mp
            with mp.get_context("fork").Pool(workers, initializer=_init, initargs=(self.file, self.root)) as pool:
                enc = pool.map(_encode_doc, args, chunksize=8)
        else:
            enc = [_encode_doc(a) for a in args]
        items = [dict(doc=d, ds=a, ss=b) for d, (a, b) in zip(docs, enc)]
        dl = np.array([sum(len(x) for x in it["ds"]) for it in items])
        msg = f"tokenised {len(items):,} documents in {time.time() - t0:.0f}s | document tokens median {int(np.median(dl)):,}, 90% {int(np.percentile(dl, 90)):,}"
        if items and items[0]["ss"] is not None:
            sl = np.array([sum(len(x) for x in it["ss"]) for it in items])
            msg += f" | summary tokens median {int(np.median(sl)):,}, 90% {int(np.percentile(sl, 90)):,}"
        log(msg)
        return items


# ======================================================================================================
# 2. Parts: document part i -> summary part i
# ======================================================================================================
def groups(lens, n):
    """contiguous split of items with token lengths `lens` into n groups of ~equal token mass -> list of index lists"""
    lens = np.asarray(lens, dtype=float); tot = max(1.0, lens.sum())
    mid = (np.cumsum(lens) - lens / 2) / tot
    g = np.minimum(n - 1, (mid * n).astype(int))
    out = [list(np.where(g == k)[0]) for k in range(n)]
    for k in range(n):                                  # never empty: borrow the nearest sentence
        if not out[k]: out[k] = [min(len(lens) - 1, int(round((k + 0.5) / n * len(lens))))]
    return out


def fit(sents, budget):
    """sentences (token arrays) -> one token list of <= budget tokens: everything if it fits, otherwise evenly spaced
    sentences across the whole part (keeps the beginning, middle and end represented)"""
    tot = sum(len(s) for s in sents)
    if tot <= budget:
        return [t for s in sents for t in s.tolist()]
    m = len(sents); k = max(1, int(m * budget / tot))
    while True:                                          # fill the budget EXACTLY (same-length prompts batch together)
        idx = np.unique(np.round(np.linspace(0, m - 1, k)).astype(int))
        out = [t for i in idx for t in sents[i].tolist()]
        if len(out) >= budget or k >= m:
            return out[:budget]
        k += max(1, k // 10)


class Layout:
    """token budgets of one model: P prompt tokens, G summary tokens per part"""

    def __init__(self, tok, arch, G, n_max=8):
        self.tok, self.arch, self.G, self.n_max = tok, arch, G, n_max
        ctx = INFO[arch][1]
        self.P = ctx if arch == "arch2_encdec" else ctx - G - 2
        self.sep = tok.enc("SUMMARY:")
        self.coef = (0.0, 1.0)

    def header(self, i, n):
        return self.tok.enc(f"Part {i + 1} of {n}.")

    def n_parts_gold(self, it):
        return int(np.clip(round(sum(len(s) for s in it["ss"]) / self.G), 1, self.n_max))

    def fit_length_model(self, items):
        """log(summary tokens) = a + b log(document tokens), fitted on training documents"""
        x = np.log([max(1, sum(len(s) for s in it["ds"])) for it in items])
        y = np.log([max(1, sum(len(s) for s in it["ss"])) for it in items])
        b, a = np.polyfit(x, y, 1); self.coef = (float(a), float(b))
        pred = np.exp(a + b * x); r = np.corrcoef(np.exp(y), pred)[0, 1]
        return dict(a=a, b=b, r=float(r))

    def n_parts_pred(self, it):
        L = max(1, sum(len(s) for s in it["ds"]))
        return int(np.clip(round(math.exp(self.coef[0] + self.coef[1] * math.log(L)) / self.G), 1, self.n_max))

    def prompts(self, it, n):
        """-> list of n prompt token lists (part header + fitted document part + separator)"""
        gs = groups([len(s) for s in it["ds"]], n)
        out = []
        for i, gi in enumerate(gs):
            h = self.header(i, n)
            body = fit([it["ds"][j] for j in gi], self.P - len(h) - len(self.sep) - (1 if self.arch == "arch2_encdec" else 0))
            out.append(h + body + (self.sep if self.arch != "arch2_encdec" else self.sep))
        return out

    def pairs(self, it):
        """training pairs (prompt, target) with the gold number of parts"""
        n = self.n_parts_gold(it)
        ps = self.prompts(it, n)
        gs = groups([len(s) for s in it["ss"]], n)
        tg = [[t for j in g for t in it["ss"][j].tolist()][: self.G - 1] + [EOS] for g in gs]
        return list(zip(ps, tg))


# ======================================================================================================
# 3. Models
# ======================================================================================================
def load_lm(arch, root, keep_layers=None, init="pretrained"):
    sys.path.insert(0, os.path.join(root, "code"))
    ck = torch.load(os.path.join(root, arch, "ckpt_best.pt"), map_location="cpu", weights_only=False)
    cfg = dict(ck["cfg"])
    if arch == "arch4_mla":
        from mla import MLAConfig, MLALM; m = MLALM(MLAConfig(**cfg))
    elif arch == "arch3_prefixlm":
        from prefixlm import PrefixLMConfig, PrefixLM; m = PrefixLM(PrefixLMConfig(**cfg))
    elif arch == "arch2_encdec":
        from encdec import EncDecConfig, EncDecLM; m = EncDecLM(EncDecConfig(**cfg))
    else:
        from model_arch import LMConfig, DecoderLM; m = DecoderLM(LMConfig(**cfg))
    if init == "pretrained":
        missing, unexpected = m.load_state_dict(ck["model"], strict=False)
        assert not unexpected and all(("rope" in x or "mask" in x or "pos_emb" in x) for x in missing), (missing[:5], unexpected[:5])
    if keep_layers:
        assert arch != "arch2_encdec"
        m.blocks = m.blocks[:keep_layers]
    return m


def layers_of(m, arch):
    return list(m.encoder) + list(m.decoder) if arch == "arch2_encdec" else list(m.blocks)


def apply_method(m, arch, cfg):
    method = cfg["method"]
    for p in m.parameters(): p.requires_grad = method == "full_ft"
    if method in ("lora", "lorma"):
        r, a = cfg.get("rank", 16), cfg.get("alpha", 16)
        make = (lambda lin: nm.LoRALinear(lin, r, a, cfg.get("lora_dropout", 0.05))) if method == "lora" else (lambda lin: nm.LoRMALinear(lin, r, a))
        n = sum(nm._replace_linears(L, make) for L in layers_of(m, arch))
        for mod in m.modules():
            if isinstance(mod, (nm.LoRALinear, nm.LoRMALinear)): mod.A.requires_grad = True; mod.B.requires_grad = True
        return f"{method} (rank {r}, alpha {a}, {n} linear layers)"
    if method in ("houlsby", "pfeiffer"):
        bott = cfg.get("bottleneck", 128); d = m.cfg.d_model; n = 0
        for L in layers_of(m, arch):
            for name in (("attn", "self_attn", "cross_attn") if method == "houlsby" else ()):
                if hasattr(L, name): setattr(L, name, nm.AdapterWrap(getattr(L, name), d, bott)); n += 1
            if hasattr(L, "ffn"): L.ffn = nm.AdapterWrap(L.ffn, d, bott); n += 1
        for mod in m.modules():
            if isinstance(mod, nm.Adapter):
                for p in mod.parameters(): p.requires_grad = True
            if isinstance(mod, nn.LayerNorm) or type(mod).__name__ == "RMSNorm":
                for p in mod.parameters(): p.requires_grad = True
        return f"{method} adapters (bottleneck {bott}, {n} adapters)"
    if method == "full_ft":
        return "full fine-tuning"
    raise ValueError(method)


def hidden_dec(m, arch, ids, plen):
    """decoder-only models: final hidden states (B,T,d)"""
    if arch == "arch3_prefixlm":
        h, _ = m._run(ids, m.prefix_mask(plen, ids.size(1))); return h
    if arch == "arch4_mla":
        return m.hidden(ids)
    x = m.tok_emb(ids)
    if m.cfg.scale_emb: x = x * math.sqrt(m.cfg.d_model)
    if m.cfg.pos_emb_type == "sinusoidal": x = x + m.pos_emb[: ids.size(1)].unsqueeze(0)
    for b in m.blocks: x = b(x)
    return m.ln_f(x)


class Objective(nn.Module):
    """returns (1, 3): summed CE, number of target tokens, summed KD loss - DataParallel-friendly"""

    def __init__(self, arch, model, teacher=None, teacher_arch=None, T=2.0):
        super().__init__()
        self.arch, self.model, self.teacher, self.tarch, self.T = arch, model, teacher, teacher_arch, T

    def forward(self, ids, plen, lens, dec_in=None, dec_lab=None):
        if self.arch == "arch2_encdec":
            with nm.autocast():
                logits = self.model.decode(self.model.encode(ids), dec_in)
            m = dec_lab != -100
            lg = logits[m].float(); tgt = dec_lab[m]
            return torch.stack([F.cross_entropy(lg, tgt, reduction="sum"), m.sum().float(), torch.zeros((), device=ids.device)])[None]
        T = ids.size(1); pos = torch.arange(T, device=ids.device)[None]
        m = (pos >= (plen[:, None] - 1)) & (pos < (lens[:, None] - 1))         # position t predicts token t+1
        tgt = ids[:, 1:][m[:, :-1]]
        with nm.autocast():
            h = hidden_dec(self.model, self.arch, ids, plen)
            lg = self.model.head(h[m]).float()
        ce = F.cross_entropy(lg, tgt, reduction="sum")
        kd = torch.zeros((), device=ids.device)
        if self.teacher is not None:
            with torch.no_grad(), nm.autocast():
                th = hidden_dec(self.teacher, self.tarch, ids, plen)
                tl = self.teacher.head(th[m]).float()
            kd = F.kl_div(F.log_softmax(lg / self.T, -1), F.softmax(tl / self.T, -1), reduction="sum") * self.T ** 2
        return torch.stack([ce, m.sum().float(), kd])[None]


def collate(batch, arch, device):
    """batch of (prompt, target) -> tensors"""
    if arch == "arch2_encdec":
        S = max(len(p) for p, _ in batch); Tt = max(len(t) for _, t in batch)
        src = torch.full((len(batch), S), PAD, dtype=torch.long); din = torch.full((len(batch), Tt), PAD, dtype=torch.long)
        lab = torch.full((len(batch), Tt), -100, dtype=torch.long)
        for i, (p, t) in enumerate(batch):
            src[i, :len(p)] = torch.tensor(p); din[i, :len(t)] = torch.tensor([BOS] + t[:-1]); lab[i, :len(t)] = torch.tensor(t)
        z = torch.zeros(len(batch), dtype=torch.long)
        return dict(ids=src.to(device), plen=z.to(device), lens=z.to(device), dec_in=din.to(device), dec_lab=lab.to(device))
    T = max(len(p) + len(t) for p, t in batch)
    ids = torch.full((len(batch), T), PAD, dtype=torch.long)
    plen = torch.zeros(len(batch), dtype=torch.long); lens = torch.zeros(len(batch), dtype=torch.long)
    for i, (p, t) in enumerate(batch):
        s = p + t; ids[i, :len(s)] = torch.tensor(s); plen[i] = len(p); lens[i] = len(s)
    return dict(ids=ids.to(device), plen=plen.to(device), lens=lens.to(device))


# ======================================================================================================
# 4. Generation
# ======================================================================================================
@torch.no_grad()
def generate(m, arch, prompts, G, device, min_frac=0.5, no_repeat=3, rep_pen=1.15, batch=16, log=None):
    """greedy decoding for a list of prompts (token lists) -> list of generated token lists (without </s>).
    Prompts of identical length are batched (KV cache for arch 3 / 4); others run one by one."""
    m.eval(); out = [None] * len(prompts)
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    buckets, cur = [], []
    for i in order:
        if cur and (len(prompts[i]) != len(prompts[cur[0]]) or len(cur) >= batch):
            buckets.append(cur); cur = []
        cur.append(i)
    if cur: buckets.append(cur)
    max_new = G if arch != "arch2_encdec" else min(G, m.cfg.dec_len - 1)
    min_new = int(min_frac * max_new); t0 = time.time()
    for bi, bk in enumerate(buckets):
        P = torch.tensor([prompts[i] for i in bk], dtype=torch.long, device=device)
        B = P.size(0); gen = [[] for _ in range(B)]; done = [False] * B
        seen = [dict() for _ in range(B)]
        with nm.autocast():
            if arch in ("arch3_prefixlm", "arch4_mla"):
                logits, cache = m.prefill(P); pos = P.size(1)
            elif arch == "arch2_encdec":
                mem = m.encode(P); dec = torch.full((B, 1), BOS, dtype=torch.long, device=device)
                logits = m.decode(mem, dec)[:, -1]
            else:
                seq = P; logits = m(seq)[:, -1]
            for step in range(max_new):
                lg = logits.float()
                for b in range(B):
                    if done[b]: continue
                    if gen[b] and rep_pen != 1.0:
                        u = torch.tensor(sorted(set(gen[b][-200:])), device=device)
                        v = lg[b, u]; lg[b, u] = torch.where(v > 0, v / rep_pen, v * rep_pen)
                    if no_repeat and len(gen[b]) >= no_repeat - 1:
                        ban = seen[b].get(tuple(gen[b][-(no_repeat - 1):]))
                        if ban: lg[b, list(ban)] = -1e9
                    lg[b, :EOS] = -1e9
                    if step < min_new: lg[b, EOS] = -1e9
                nxt = lg.argmax(-1)
                for b in range(B):
                    if done[b]: continue
                    t = int(nxt[b])
                    if t == EOS: done[b] = True; continue
                    if no_repeat and len(gen[b]) >= no_repeat - 1:
                        seen[b].setdefault(tuple(gen[b][-(no_repeat - 1):]), set()).add(t)
                    gen[b].append(t)
                if all(done): break
                tok = nxt[:, None]
                if arch in ("arch3_prefixlm", "arch4_mla"):
                    logits, cache = m.decode_step(tok, cache, pos); pos += 1
                elif arch == "arch2_encdec":
                    dec = torch.cat([dec, tok], 1); logits = m.decode(mem, dec)[:, -1]
                else:
                    seq = torch.cat([seq, tok], 1)[:, -1024:]; logits = m(seq)[:, -1]
        for b, i in enumerate(bk): out[i] = gen[b]
        if log and (bi == 0 or (bi + 1) % 50 == 0):
            done_n = sum(len(x) for x in buckets[: bi + 1])
            log(f"  generated {done_n:,}/{len(prompts):,} parts | {(time.time() - t0) / 60:.1f} min")
    return out


def dedupe(text):
    """drop repeated sentences (exact or >= 80% word overlap with an earlier one)"""
    keep, seen = [], []
    for s in sentences(text, min_chars=1):
        w = set(re.findall(r"\w+", s.lower()))
        if not w: continue
        if any(len(w & v) / max(1, min(len(w), len(v))) >= 0.8 for v in seen): continue
        keep.append(s); seen.append(w)
    return " ".join(keep)


def summarise(m, arch, lay, items, device, n_mode="pred", log=print, **gk):
    """documents -> generated summary texts (parts concatenated, duplicates removed)"""
    jobs, owner = [], []
    for k, it in enumerate(items):
        n = lay.n_parts_pred(it) if n_mode == "pred" else (1 if n_mode == "single" else lay.n_parts_gold(it))
        for p in lay.prompts(it, n):
            jobs.append(p); owner.append(k)
    gen = generate(m, arch, jobs, lay.G, device, log=log, **gk)
    parts = [[] for _ in items]
    for k, g in zip(owner, gen): parts[k].append(lay.tok.dec(g))
    return [dedupe(" ".join(p)) or "Summary not available." for p in parts]


# ======================================================================================================
# 5. Metrics (no pretrained model is used locally; the leaderboard computes BERTScore)
# ======================================================================================================
_W = re.compile(r"[a-z0-9]+")


def _ngrams(w, n):
    return Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1))


def _lcs(a, b):
    if len(a) > 1200: a = a[:1200]
    if len(b) > 1200: b = b[:1200]
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge(preds, refs):
    r1, r2, rl, rec1 = [], [], [], []
    for p, r in zip(preds, refs):
        a, b = _W.findall(p.lower()), _W.findall(r.lower())
        for n, acc in ((1, r1), (2, r2)):
            pa, rb = _ngrams(a, n), _ngrams(b, n); ov = sum((pa & rb).values())
            P = ov / max(1, sum(pa.values())); R = ov / max(1, sum(rb.values()))
            acc.append(2 * P * R / max(1e-9, P + R))
            if n == 1: rec1.append(R)
        l = _lcs(a, b); P = l / max(1, min(len(a), 1200)); R = l / max(1, min(len(b), 1200))
        rl.append(2 * P * R / max(1e-9, P + R))
    return dict(rouge1=float(np.mean(r1)), rouge2=float(np.mean(r2)), rougeL=float(np.mean(rl)), rouge1_recall=float(np.mean(rec1)))


def redundancy(texts):
    """share of repeated word 3-grams inside each text (0 = no repetition)"""
    v = []
    for t in texts:
        w = _W.findall(t.lower()); g = _ngrams(w, 3); tot = sum(g.values())
        v.append(0.0 if not tot else 1 - len(g) / tot)
    return float(np.mean(v))


def text_stats(preds, refs, docs):
    wp = np.array([len(p.split()) for p in preds]); wr = np.array([len(r.split()) for r in refs]) if refs else None
    wd = np.array([len(d.split()) for d in docs])
    s = dict(pred_words=float(wp.mean()), compression=float((wp / np.maximum(1, wd)).mean()), redundancy=redundancy(preds))
    if refs: s["ref_words"] = float(wr.mean())
    return s


# ======================================================================================================
# 6. Training
# ======================================================================================================
def trainable_state(m):
    return {k: v.detach().cpu() for k, v in m.state_dict().items() if k in {n for n, p in m.named_parameters() if p.requires_grad}}


def train(cfg, model, arch, tr_pairs, va_pairs, device, out_dir, log=print, teacher=None, teacher_arch=None):
    obj = Objective(arch, model, teacher, teacher_arch, cfg.get("kd_T", 2.0))
    gpus = torch.cuda.device_count()
    run = nn.DataParallel(obj) if gpus > 1 else obj
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    rng = random.Random(cfg["seed"]); B, acc = cfg["batch"], cfg["accum"]
    spe = math.ceil(len(tr_pairs) / (B * acc)); total = int(spe * cfg["epochs"]); warm = max(1, int(0.05 * total))
    alpha = cfg.get("kd_alpha", 1.0) if teacher is not None else 1.0
    log(f"train pairs {len(tr_pairs):,} | val pairs {len(va_pairs):,} | {spe:,} steps/epoch x {cfg['epochs']} | trainable "
        f"{sum(p.numel() for p in params):,} | GPUs {gpus}" + (f" | KD alpha {alpha} T {cfg.get('kd_T', 2.0)}" if teacher is not None else ""))
    assert gpus >= 1, "NO GPU - set Accelerator to GPU T4 x2"

    def val_loss():
        model.eval(); s = n = 0.0
        with torch.no_grad():
            for i in range(0, len(va_pairs), B * 2):
                o = run(**collate(va_pairs[i:i + B * 2], arch, device)).sum(0); s += float(o[0]); n += float(o[1])
        model.train(); return s / max(1, n)

    best, best_state, step, t0, hist = 1e9, None, 0, time.time(), []
    budget = cfg.get("budget_hours", 6.0) * 3600
    model.train(); stop = False
    for ep in range(math.ceil(cfg["epochs"])):
        order = list(range(len(tr_pairs))); rng.shuffle(order); run_l = run_n = 0.0; gnorm = []
        for k in range(0, len(order), B):
            o = run(**collate([tr_pairs[i] for i in order[k:k + B]], arch, device)).sum(0)
            ce, n, kd = o[0], o[1].clamp(min=1), o[2]
            loss = (alpha * ce + (1 - alpha) * kd) / n if teacher is not None else ce / n
            scaler.scale(loss / acc).backward(); run_l += float(ce); run_n += float(n)
            if (k // B + 1) % acc: continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = cfg["lr"] * f
            scaler.unscale_(opt); gnorm.append(float(torch.nn.utils.clip_grad_norm_(params, 1.0)))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if step >= total or time.time() - t0 > budget: stop = True; break
        v = val_loss(); hist.append(dict(epoch=ep + 1, train_loss=run_l / max(1, run_n), val_loss=v))
        note = ""
        if v < best:
            best, note = v, "  <- best"; best_state = trainable_state(model)
            torch.save(best_state, os.path.join(out_dir, f"p6_trainable_{cfg['run']}.pt"))
        log(f"epoch {ep + 1}: train loss {run_l / max(1, run_n):.4f} | val loss {v:.4f} (ppl {math.exp(min(20, v)):.1f}){note} | "
            f"grad norm {np.mean([g for g in gnorm if math.isfinite(g)] or [0]):.3f} | {(time.time() - t0) / 60:.0f} min")
        if stop:
            if time.time() - t0 > budget: log("time budget reached - stopping")
            break
    model.load_state_dict(best_state, strict=False)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    return dict(best_val_loss=best, history=hist, train_minutes=round((time.time() - t0) / 60, 1), peak_gpu_gb=round(peak, 2))


def build(cfg, root):
    m = load_lm(cfg["arch"], root, cfg.get("keep_layers"), cfg.get("init", "pretrained"))
    desc = apply_method(m, cfg["arch"], cfg)
    return m, desc


def load_teacher(cfg_t, root, path, device):
    m, _ = build(cfg_t, root)
    st = torch.load(path, map_location="cpu", weights_only=False)
    missing, unexpected = m.load_state_dict(st, strict=False)
    assert not unexpected, unexpected[:5]
    for p in m.parameters(): p.requires_grad = False
    return m.to(device).eval()


def run_experiment(cfg, D, device, out_root, log=print, teacher=None, teacher_arch=None):
    t_start = time.time(); out_dir = os.path.join(out_root, cfg["run"]); os.makedirs(out_dir, exist_ok=True)
    tok = Tok(cfg["arch"], D["root"])
    T = {s: tok.docs(D[s], workers=D.get("workers", 4), log=log) for s in ("train", "val", "test")}
    lay = Layout(tok, cfg["arch"], cfg["G"], cfg.get("n_max", 8))
    lm = lay.fit_length_model(T["train"])
    log(f"layout: prompt {lay.P} tokens, summary part {lay.G} tokens | length model: summary ~ exp({lm['a']:.2f}) * doc^{lm['b']:.2f} (r {lm['r']:.2f})")
    tr = [p for it in T["train"] for p in lay.pairs(it)]; va = [p for it in T["val"] for p in lay.pairs(it)]
    nparts = Counter(lay.n_parts_gold(it) for it in T["train"])
    log(f"parts per training document: {dict(sorted(nparts.items()))}")
    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"]); np.random.seed(cfg["seed"])
    model, desc = build(cfg, D["root"]); model.to(device)
    total = sum(p.numel() for p in model.parameters()); trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"\n=== {cfg['run']} | {cfg['arch']} | {desc} | params {total:,}, trainable {trainable:,} ({trainable / total:.2%})"
        + (f" | KD from {teacher_arch}" if teacher is not None else ""))
    res = train(cfg, model, cfg["arch"], tr, va, device, out_dir, log, teacher, teacher_arch)
    teacher = None; torch.cuda.empty_cache()
    gk = dict(min_frac=cfg.get("min_frac", 0.5), no_repeat=cfg.get("no_repeat", 3), rep_pen=cfg.get("rep_pen", 1.15),
              batch=cfg.get("gen_batch", 16))
    nv = cfg.get("n_val_gen", 60); vi = T["val"][:nv]; refs = [it["doc"]["summary"] for it in vi]
    log(f"\ngenerating validation summaries ({len(vi)} documents)...")
    t0 = time.time(); vp = summarise(model, cfg["arch"], lay, vi, device, "pred", log=log, **gk); vt = time.time() - t0
    ev = dict(**rouge(vp, refs), **text_stats(vp, refs, [it["doc"]["document"] for it in vi]), sec_per_doc=round(vt / max(1, len(vi)), 2))
    log("VALIDATION (" + ", ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in ev.items()) + ")")
    if cfg.get("ablate_single"):
        sp = summarise(model, cfg["arch"], lay, vi, device, "single", log=None, **gk)
        ev1 = dict(**rouge(sp, refs), **text_stats(sp, refs, [it["doc"]["document"] for it in vi]))
        log("ABLATION single pass (whole document compressed into one prompt): " + ", ".join(f"{k} {v:.4f}" for k, v in ev1.items()))
        ev["single_pass"] = ev1
    json.dump([dict(id=it["doc"]["id"], reference=r, generated=p) for it, r, p in zip(vi, refs, vp)],
              open(os.path.join(out_dir, "val_generations.json"), "w"), ensure_ascii=False, indent=1)
    log(f"example:\n  REFERENCE: {refs[0][:600]}\n  GENERATED: {vp[0][:600]}")
    log(f"\ngenerating test summaries ({len(T['test'])} documents)...")
    t0 = time.time(); tp = summarise(model, cfg["arch"], lay, T["test"], device, "pred", log=log, **gk); tt = time.time() - t0
    zp = write_submission(tp, D, out_dir, cfg["run"], log)
    rec = dict(run=cfg["run"], arch=cfg["arch"], method=desc, cfg=cfg, params_total=total, params_trainable=trainable,
               val=ev, test_minutes=round(tt / 60, 1), zip=os.path.basename(zp), minutes_total=round((time.time() - t_start) / 60, 1), **res)
    json.dump(rec, open(os.path.join(out_dir, "record.json"), "w"), indent=1, default=str)
    del model; torch.cuda.empty_cache()
    return rec


def write_submission(summaries, D, out_dir, name, log=print):
    sub = [dict(id=d["id"], summary=s) for d, s in zip(D["test"], summaries)]
    assert len(sub) == len(D["test"]) and all(isinstance(s["summary"], str) and s["summary"].strip() for s in sub)
    assert len({json.dumps(s["id"]) for s in sub}) == len(sub)
    os.makedirs(out_dir, exist_ok=True)
    pj = os.path.join(out_dir, "prediction.json"); json.dump(sub, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
    log(f"{name}: {len(sub):,} test summaries, {np.mean([len(s['summary'].split()) for s in sub]):.0f} words on average -> {zp}")
    return zp

In [ ]:
import importlib
sys.path.insert(0, os.path.abspath("p6src"))
import ner_data as nd, ner_model as nm, sm
for _m in (nd, nm, sm): importlib.reload(_m)
assert getattr(sm, "VERSION", "") == "p6-v1", "OLD CODE - import the Phase-6 notebook file"
print("Phase-6 code imported | code version:", sm.VERSION)

In [ ]:
# 4 · Load Phase_6_Summ: train (+ a held-out validation split made from it) / test, with checks
from collections import Counter
from huggingface_hub import HfApi, hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
files = [f for f in HfApi(token=HF_TOKEN).list_repo_files(REPO, repo_type="dataset") if f.startswith("Phase_6")]
print("\n".join(files))
get = lambda n: hf_hub_download(REPO, next(f for f in files if f.endswith("/" + n)), repo_type="dataset", token=HF_TOKEN)
train_all = sm.parse(sm.read_json_any(get("train.json")))
test = sm.parse(sm.read_json_any(get("test.json")))
assert all(d["summary"] for d in train_all), "every training document needs a summary"
assert len({json.dumps(d["id"]) for d in test}) == len(test), "duplicate test ids"
rnd = random.Random(0); idx = list(range(len(train_all))); rnd.shuffle(idx)
n_val = int(min(300, max(50, 0.05 * len(train_all))))
val = [train_all[i] for i in sorted(idx[:n_val])]; train = [train_all[i] for i in sorted(idx[n_val:])]
D = dict(root=ROOT, workers=min(4, os.cpu_count() or 1), train=train, val=val, test=test)
for k in ("train", "val", "test"):
    w = np.array([len(d["document"].split()) for d in D[k]])
    s = np.array([len(d["summary"].split()) for d in D[k] if d["summary"]]) if D[k][0]["summary"] else None
    print(f"{k:<5}: {len(D[k]):5,} documents | document words median {int(np.median(w)):,}, 90% {int(np.percentile(w, 90)):,}, max {int(w.max()):,}"
          + (f" | summary words median {int(np.median(s)):,}, 90% {int(np.percentile(s, 90)):,} | compression {np.median(s / np.maximum(1, w[:len(s)])):.3f}" if s is not None else ""))
pre = lambda d: d["document"][:1500]
tr_p = {pre(d) for d in train_all}
print("OVERLAP CHECK | test documents whose start also appears in train:", sum(pre(d) in tr_p for d in test),
      "| test documents carrying a summary in test.json:", sum(1 for d in test if d["summary"]), "(never used)")
ex = train[0]
print(f"\nexample {ex['id']}\n  DOCUMENT: {ex['document'][:400]}\n  SUMMARY : {ex['summary'][:400]}")
D_SMOKE = dict(D, train=train[:12], val=val[:3], test=test[:3])

In [ ]:
TEACHER, TEACHER_ARCH = None, None

In [ ]:
# 6 · Smoke check: every run on 12 training / 3 validation / 3 test documents, 1 short epoch, short generation
if SMOKE_CHECK:
    t0 = time.time()
    for c in PLAN:
        sm.run_experiment(dict(c, run="smoke_" + c["run"], epochs=1, budget_hours=0.05, n_val_gen=2, G=min(c["G"], 32)),
                          D_SMOKE, DEVICE, OUT + "/smoke", teacher=TEACHER if c.get("kd") else None,
                          teacher_arch=TEACHER_ARCH if c.get("kd") else None)
    gc.collect(); torch.cuda.empty_cache()
    print(f"\nSMOKE CHECK PASSED in {(time.time() - t0) / 60:.1f} min (numbers mean nothing)")

In [ ]:
# 7 · The real runs
t0 = time.time(); records = []
for c in PLAN:
    records.append(sm.run_experiment(c, D, DEVICE, OUT, teacher=TEACHER if c.get("kd") else None,
                                     teacher_arch=TEACHER_ARCH if c.get("kd") else None))
    gc.collect(); torch.cuda.empty_cache()
    json.dump(records, open(os.path.join(OUT, "summary.json"), "w"), indent=1, default=str)
print(f"all runs finished in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# 8 · Summary + what to submit
for r in records:
    v = r["val"]
    print(f"\n{r['run']} | {r['method']} | params {r['params_total']:,}, trainable {r['params_trainable']:,} | best val loss {r['best_val_loss']:.4f} | "
          f"train {r['train_minutes']:.0f} min, test generation {r['test_minutes']:.0f} min, peak GPU {r['peak_gpu_gb']} GB")
    print(f"   validation: ROUGE-1 {v['rouge1']:.4f}, ROUGE-2 {v['rouge2']:.4f}, ROUGE-L {v['rougeL']:.4f}, ROUGE-1 recall (coverage) {v['rouge1_recall']:.4f} | "
          f"{v['pred_words']:.0f} words vs reference {v.get('ref_words', 0):.0f} | repeated 3-grams {v['redundancy']:.3f} | {v['sec_per_doc']} s/document")
    if "single_pass" in v:
        s = v["single_pass"]
        print(f"   single-pass ablation: ROUGE-1 {s['rouge1']:.4f}, ROUGE-L {s['rougeL']:.4f}, coverage {s['rouge1_recall']:.4f}, {s['pred_words']:.0f} words")
print("\nSUBMIT:", os.path.join(OUT, records[0]["run"], records[0]["zip"]))